## Step 1: Import Libraries & API Keys

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is missing.")

/Users/daniellechoi/anaconda3/envs/ai-env313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2: Set up Pushover

In [2]:
load_dotenv()

True

In [3]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "http://api.pushover.net/1/messages.json"

In [4]:
if pushover_user is None:
    raise Exception("User is missing.")
if pushover_token is None:
    raise Exception("Pushover token is missing.")

In [7]:
# Test pushover
import requests

def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
send_notification("Hello to myself, from this AI Engineering training.")

## Step 3: Describe Pushover as an LLM tool

In [8]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the user's phone via Pushover. Use this to alert the user about important events, completed tasks, or time-sensitive information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

## Step 4: Add Pushover to the list of tools for the LLM

In [9]:
tools = [{
    "type": "function",
    "function": send_notification_function
}]

## Step 5: Calling the tool from an LLM

In [10]:
client = OpenAI()
response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = [{
        "role": "user", "content": "Please send me a notification telling me what amazing progress \
        I am making on the AI Engineering training."
    }],
    tools = tools,
    tool_choice="auto" # none, required, {}, auto
)

# Check if model wants to call a tool
message = response.choices[0].message

In [11]:
print(message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_mTT5Qbb91uWpAluLL1VqeF81', function=Function(arguments='{"message":"You are making amazing progress on the AI Engineering training. Keep up the great work!"}', name='send_notification'), type='function')])


In [12]:
if message.tool_calls:
    tool_call = message.tool_calls[0]
    import json
    args = json.loads(tool_call.function.arguments)

    # Actually send the notification
    send_notification(args["message"])
    print(f"Send notification: {args["message"]}")

else:
    print(message.content)

Send notification: You are making amazing progress on the AI Engineering training. Keep up the great work!
